# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load and explore the [Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells](https://sen.science/doi/10.71728/senscience.vq4a-28xa/) dataset using the `mlcroissant` library in Python. You'll learn to examine its Croissant structure, access record sets, fields, and analyze the protein abundance data programmatically.

### Dataset Source
The dataset is described by a Croissant schema at:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}\n")

## 2. Data Overview
Review available record sets, their `@id`s, associated fields, and columns.
All Croissant entities are referenced by their unique `@id`.

In [ ]:
# List available record sets in the dataset
print('Available record sets:')
record_sets = metadata.record_sets
for rs in record_sets:
    print(f"  - @id: {rs.id}, name: {rs.name}")
    # List fields of this record set
    if hasattr(rs, 'fields') and rs.fields:
        print("    Fields:")
        for field in rs.fields:
            field_str = f"      - @id: {field.id}, name: {field.name}, dataType: {getattr(field, 'data_type', 'N/A')}"
            print(field_str)
    # List columns (correspond to table columns or file columns)
    if hasattr(rs, 'columns') and rs.columns:
        print("    Columns:")
        for col in rs.columns:
            print(f"      - @id: {col.id}, name: {col.name}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame.

**Choose a record set and fields based on the overview above. All references use their `@id` property.**

In [ ]:
# For this dataset, there's typically a single tabular record set containing the main proteomics table.
# You can update `main_recordset_id` below if reviewing another record set.

# Example: Find the main table record set (by name or id):
main_recordset_id = None
main_recordset_name = ""  # Optionally match by name if available
for rs in metadata.record_sets:
    # Example: look for 'protein' keyword in the record set name
    if hasattr(rs, 'name') and ('protein' in rs.name.lower() or 'main' in rs.name.lower()):
        main_recordset_id = rs.id
        main_recordset_name = rs.name
        break

# If not found above, pick the first record set
if main_recordset_id is None and metadata.record_sets:
    main_recordset_id = metadata.record_sets[0].id
    main_recordset_name = metadata.record_sets[0].name

print(f"Using record set: {main_recordset_id} ({main_recordset_name})\n")

dataframes = {}
# We'll extract all record sets for inspection
for rs in metadata.record_sets:
    rec_id = rs.id
    records = list(dataset.records(record_set=rec_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[rec_id] = df
        print(f"Loaded DataFrame for {rec_id} with shape {df.shape}")

if main_recordset_id in dataframes:
    print(f"Columns in main record set ({main_recordset_id}):")
    print(dataframes[main_recordset_id].columns.tolist())
    display(dataframes[main_recordset_id].head())
else:
    print(f"No records loaded for record set {main_recordset_id}")

## 4. Exploratory Data Analysis (EDA)
Let's process and explore the protein abundance dataset. We'll:
- Filter on records with high values in a numeric field (e.g., MW or Coverage [%]).
- Normalize a numeric field.
- Group by a categorical field such as 'description' or 'accession', if present.

*Note:* All fields are referenced with their `@id` as shown in the overview.

In [ ]:
# Select a numeric field for analysis—find a likely candidate:
main_df = dataframes.get(main_recordset_id, pd.DataFrame())
numeric_field_id = None
# Try to identify a numeric column by common proteomics field names
candidate_names = ['MW', 'coverage', 'Coverage [%]', 'peptides', 'PSM', 'score', 'Abundance']
for col in main_df.columns:
    lowered = col.lower()
    if any(name.lower() in lowered for name in candidate_names):
        if pd.api.types.is_numeric_dtype(main_df[col]):
            numeric_field_id = col
            break
# Fallback: Pick any numeric dtype column
if numeric_field_id is None:
    for col in main_df.columns:
        if pd.api.types.is_numeric_dtype(main_df[col]):
            numeric_field_id = col
            break

group_field_id = None
for col in main_df.columns:
    if 'accession' in col.lower() or 'protein' in col.lower() or 'description' in col.lower():
        group_field_id = col
        break

if numeric_field_id is not None and not main_df.empty:
    print(f"Using numeric field: {numeric_field_id}")
    threshold = main_df[numeric_field_id].mean()  # use mean as a logical threshold

    filtered_df = main_df[main_df[numeric_field_id] > threshold].copy()
    print(f"Filtered records with {numeric_field_id} > {threshold:.2f} (mean):")
    display(filtered_df.head())

    # Normalization
    norm_col = f"{numeric_field_id}_normalized"
    filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"First 5 normalized values for {numeric_field_id}:")
    display(filtered_df[[numeric_field_id, norm_col]].head())

    # Group by group_field_id if available & is not unique value per row
    if group_field_id and group_field_id in filtered_df.columns and filtered_df[group_field_id].nunique() < len(filtered_df):
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().sort_values(ascending=False)
        print(f"Grouped mean of {numeric_field_id} by {group_field_id} (top rows):")
        display(grouped_df.head())
else:
    print("Could not find a suitable numeric field for EDA or an empty DataFrame.")

## 5. Visualization
Visualize data distributions and relationships between fields using `matplotlib`.

*The following example shows histograms and scatter plots for selected fields (`@id`-referenced columns):*

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if not main_df.empty and numeric_field_id:
    plt.figure(figsize=(8, 4))
    sns.histplot(main_df[numeric_field_id].dropna(), bins=30, kde=True, color='b')
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.tight_layout()
    plt.show()

    # Scatter plot of two numeric fields if possible
    # Try to find a secondary numeric column
    numeric_cols = [col for col in main_df.columns if pd.api.types.is_numeric_dtype(main_df[col]) and col != numeric_field_id]
    if numeric_cols:
        plt.figure(figsize=(6, 6))
        sns.scatterplot(
            data=main_df, x=numeric_field_id, y=numeric_cols[0], alpha=0.7
        )
        plt.title(f"Scatter plot of {numeric_field_id} vs {numeric_cols[0]}")
        plt.xlabel(numeric_field_id)
        plt.ylabel(numeric_cols[0])
        plt.tight_layout()
        plt.show()
else:
    print("Visualization skipped due to missing numeric columns or data.")

## 6. Conclusion
This notebook demonstrated how to efficiently explore a FAIR Croissant dataset with `mlcroissant`:
- Loaded and inspected metadata and record sets by `@id`
- Extracted tabular proteomics data and identified core numeric fields
- Applied simple EDA, including filtering and normalization
- Provided visual summaries for protein features

For more advanced analysis, consult the dataset [documentation](https://sen.science/doi/10.71728/senscience.vq4a-28xa/) and expand on these operations.